<style>
    .input_area, .output_area, pre, code {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
    }
</style>

In [30]:
# Import libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns 

import sqlite3 as db

import sys
sys.path.append("../src")
from trainTestSplit import get_reg_splits, modified_reg_splits

import scipy.stats as st
import statsmodels.api as sm 
import pylab as py 

from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import RFE

ImportError: cannot import name 'modified_reg_splits' from 'trainTestSplit' (c:\Users\ejtes\OneDrive\Desktop\Class Notes & Assignments\2026 Spring\DTSC 2301 & 2302\2302\homework\project3\DTSC2302-Project3\notebooks\../src\trainTestSplit.py)

Load the dataset

In [25]:
# Load data
conn = db.connect("../data/charlotte_housing.db")

master_df = pd.read_sql_query("SELECT * FROM master", conn)

conn.close()

print(f"Data loaded successfully. Master dataset shape:, {master_df.shape[0]} records, {master_df.shape[1]} features.")
display(master_df.head())


Data loaded successfully. Master dataset shape:, 458 records, 23 features.


,NPA,household_income,home_ownership,rental_houses,food_nutrition,employment,housing_age,code_violations,foreclosures,new_constructions,...,race_white,race_hispanic,race_asian,race_other,transit_proximity,financial_proximity,grocery_proximity,housing_cost_burden,displacement_risk,crime_total
0,2,-0.407371,-0.685002,-0.163924,0.240595,0.088074,1.908974,0.229973,-0.351833,-0.303270,...,0.571876,-0.392703,-0.618768,1.897747,0.242736,-0.292939,0.376054,0.278606,0,0.581649
1,3,0.436290,-0.677445,1.314855,-0.957022,0.482288,2.089046,0.106927,-0.351833,1.549196,...,1.265442,-0.812366,-0.224057,-0.070897,9.043236,10.992720,9.194680,0.284134,0,3.480599
2,4,3.586770,1.531572,-0.911621,-1.108914,0.555413,0.408373,-0.631352,-0.351833,-0.284652,...,1.606091,-1.007781,-0.260578,-1.391047,-0.560484,-0.551553,-0.673864,0.151362,0,-0.618596
3,5,-0.868820,-1.245985,-0.172231,1.777049,-1.885547,1.368758,0.845205,-0.351833,0.134247,...,-1.197404,-0.254952,-0.697429,-0.009136,-0.578973,-0.546764,-0.750563,0.313884,1,-0.264850
4,6,-1.107639,-0.946507,1.937935,1.169478,0.914241,1.728902,1.337391,5.110524,-0.154328,...,-1.057031,-0.429544,-0.791542,-0.541828,-0.078758,0.062656,-0.143792,0.633077,1,1.814352


Train/Test Split

In [26]:
X_train, X_test, y_train, y_test = get_reg_splits(master_df)
print(f"Training set: {X_train.shape[0]} records, {X_train.shape[1]} features.")
print(f"Testing set: {X_test.shape[0]} records, {X_test.shape[1]} features.")

print("\n====Training set features:====\n")
for col in X_train.columns:
    print(f"{col}: {X_train[col].dtype}")

Training set: 343 records, 17 features.
Testing set: 115 records, 17 features.

====Training set features:====

household_income: float64
home_ownership: float64
rental_houses: float64
food_nutrition: float64
employment: float64
housing_age: float64
code_violations: float64
foreclosures: float64
new_constructions: float64
age_of_residents: float64
race_black: float64
race_hispanic: float64
race_asian: float64
race_other: float64
transit_proximity: float64
grocery_proximity: float64
crime_total: float64


Features with high VIF and Multicollinearity were already identified and handled during the data prep/EDA process. These features were dropped when setting up the multiple linear regression UDF for train test split. To expedite the feature selection, a Recursive Feature Elimination (RFE) will be ran to find the best combination of features based on Adjusted R². This is because RFE is best suited for a Multiple Linear Regression as it considers features that are individually weak, but collectively strong predictors... or individually strong but redundant with others.

In [ ]:
lr = LinearRegression()

for k in range(1, X_train.shape[1] + 1):
    rfe = RFE(estimator=lr, n_features_to_select=k)
    rfe.fit(X_train, y_train)

    X_selected = rfe.transform(X_train)

    r2_scores = cross_val_score(lr, X_selected, y_train, cv=8, scoring="r2")
    mean_r2 = r2_scores.mean()
    n, p = X_train.shape[0], k
    adj_r2 = 1 - (1 - mean_r2) * (n - 1) / (n - p - 1)
    
    selected = X_train.columns[rfe.support_].tolist()
    print(f"\nk={k}: Adj R²={adj_r2:.4f}")
    print(f"  Features: {selected}")


k=1: Adj R²=0.5919
  Features: ['home_ownership']

k=2: Adj R²=0.6911
  Features: ['household_income', 'home_ownership']

k=3: Adj R²=0.7085
  Features: ['household_income', 'home_ownership', 'food_nutrition']

k=4: Adj R²=0.7122
  Features: ['household_income', 'home_ownership', 'food_nutrition', 'age_of_residents']

k=5: Adj R²=0.7137
  Features: ['household_income', 'home_ownership', 'food_nutrition', 'age_of_residents', 'race_asian']

k=6: Adj R²=0.7125
  Features: ['household_income', 'home_ownership', 'food_nutrition', 'new_constructions', 'age_of_residents', 'race_asian']

k=7: Adj R²=0.7095
  Features: ['household_income', 'home_ownership', 'food_nutrition', 'new_constructions', 'age_of_residents', 'race_asian', 'crime_total']

k=8: Adj R²=0.7070
  Features: ['household_income', 'home_ownership', 'food_nutrition', 'new_constructions', 'age_of_residents', 'race_asian', 'grocery_proximity', 'crime_total']

k=9: Adj R²=0.7061
  Features: ['household_income', 'home_ownership', 'fo

`race_asian` was selected by RFE multiple times. However, there's no theoretical justification for why it should be included. Also, given the sensitive nature of this project, I'm going to try to exclude it and run the RFE analysis again.

In [23]:
# Remove race_asian from X_train temporarily
X_train = X_train.drop(columns=["race_asian"])

lr = LinearRegression()
results = []

for k in range(1, X_train.shape[1] + 1):
    rfe = RFE(estimator=lr, n_features_to_select=k)
    rfe.fit(X_train, y_train)
    X_selected = rfe.transform(X_train)
    r2_scores = cross_val_score(lr, X_selected, y_train, cv=5, scoring="r2")
    mean_r2 = r2_scores.mean()
    n, p = X_train.shape[0], k
    adj_r2 = 1 - (1 - mean_r2) * (n - 1) / (n - p - 1)
    selected = X_train.columns[rfe.support_].tolist()
    print(f"k={k}: Adj R²={adj_r2:.4f} | Features: {selected}")

k=1: Adj R²=0.5981 | Features: ['home_ownership']
k=2: Adj R²=0.6989 | Features: ['household_income', 'home_ownership']
k=3: Adj R²=0.7143 | Features: ['household_income', 'home_ownership', 'food_nutrition']
k=4: Adj R²=0.7194 | Features: ['household_income', 'home_ownership', 'food_nutrition', 'age_of_residents']
k=5: Adj R²=0.7185 | Features: ['household_income', 'home_ownership', 'food_nutrition', 'age_of_residents', 'grocery_proximity']
k=6: Adj R²=0.7182 | Features: ['household_income', 'home_ownership', 'food_nutrition', 'age_of_residents', 'transit_proximity', 'grocery_proximity']
k=7: Adj R²=0.7187 | Features: ['household_income', 'home_ownership', 'food_nutrition', 'new_constructions', 'age_of_residents', 'transit_proximity', 'grocery_proximity']
k=8: Adj R²=0.7160 | Features: ['household_income', 'home_ownership', 'food_nutrition', 'new_constructions', 'age_of_residents', 'race_black', 'transit_proximity', 'grocery_proximity']
k=9: Adj R²=0.7160 | Features: ['household_income

After removing `race_asian`, the possible features associated with the highest Adj R² are more plausible explanations for `housing_cost_burden`.   
Adj R² also starts to decrease once more than four features are added. K-fold 5 includes:    
- `household_income` is an obvious direct measure of being able to afford housing.  
- `home_ownership` gives us insight on the stability of the population in the neighborhood
- `food_nutrition` is a proxy variable for poverty as a high food nutrition services is usually associated with lower income which equals higher burden
- `age_of_residents` older residents tend to be long term home owners with lower mortgage rates. Less younger populations tend to be an indicator of a dwindling quality of life.

However, we believe that the features for k=7, while having a slightly lower score, can tell a richer story.
- `new_construction` could indicate gentrification. Especially given Charlotte's population growth over the decades.  
- `transit_proximity` access to public transportation can affect housing costs. Too close and you're paying a lot for convenience. Too far makes it difficult to get around, especially commuting to a job.
- `grocery_proximity` food access, can be a proxy for urban development and population density.
  
The difference in scores is negligible. We think the penalty in the score is worth it for the addtional insight. 
A new train/test split will be created with these features in mind.


New Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = modified_reg_splits(master_df)